# Partie 0 — Premières règles avec Apriori

Dans `order_data.csv`, une ligne représente un panier et chaque espace sépare deux articles. On conserve l’orthographe du fichier pour reproduire l’exercice, y compris `cheesse` et `cerals`.

In [1]:
from pathlib import Path

data_dir = Path("data") if Path("data").is_dir() else Path("../data")
with (data_dir / "order_data.csv").open(encoding="utf-8-sig") as orders:
    demo_transactions = [line.split() for line in orders if line.strip()]

print(f"{len(demo_transactions)} paniers. Premier panier : {demo_transactions[0]}")

20 paniers. Premier panier : ['toothpaste', 'brush', 'milk', 'cereals', 'honey', 'bread', 'butter', 'cheese', 'yogurt']


## Extraction des règles

Un support de 25 % exige au moins 5 paniers sur les 20. La confiance minimale est de 20 % et le lift minimal de 2. Le filtre sur la taille de l’itemset est aussi appliqué à la sortie pour garantir au moins deux articles par règle.

In [2]:
from apyori import apriori

results = apriori(
    demo_transactions,
    min_support=0.25,
    min_confidence=0.2,
    min_lift=2,
    min_length=2,
)

rules = []
for result in results:
    if len(result.items) < 2:
        continue
    for stats in result.ordered_statistics:
        if stats.items_base and stats.items_add:
            left = ", ".join(sorted(stats.items_base))
            right = ", ".join(sorted(stats.items_add))
            rules.append((left, right, result.support, stats.confidence, stats.lift))

rules.sort(key=lambda rule: (-rule[2], -rule[4], -rule[3], rule[0], rule[1]))
# On privilégie les règles à deux articles pour cette première lecture.
pairs = [rule for rule in rules if ", " not in rule[0] and ", " not in rule[1]]
print(f"{len(rules)} règles au total, dont {len(pairs)} règles à deux articles.")
other_rules = [rule for rule in rules if rule not in pairs]
for left, right, support, confidence, lift in (pairs + other_rules)[:5]:
    print(f"{{{left}}} → {{{right}}} | support {support:.0%} | "
          f"confiance {confidence:.1%} | lift {lift:.2f}")

12 règles au total, dont 4 règles à deux articles.
{mouthwash} → {toothpaste} | support 30% | confiance 85.7% | lift 2.14
{toothpaste} → {mouthwash} | support 30% | confiance 75.0% | lift 2.14
{brush} → {toothpaste} | support 25% | confiance 100.0% | lift 2.50
{toothpaste} → {brush} | support 25% | confiance 62.5% | lift 2.50
{milk, shampoo} → {cereals} | support 25% | confiance 100.0% | lift 2.50


### Résultats vérifiés par comptage des paniers

12 règles respectent les seuils. Voici les quatre règles à deux articles et une règle à trois articles.

| Règle | Support | Confiance | Lift |
| --- | ---: | ---: | ---: |
| {mouthwash} → {toothpaste} | 30% | 85.7% | 2.14 |
| {toothpaste} → {mouthwash} | 30% | 75.0% | 2.14 |
| {brush} → {toothpaste} | 25% | 100.0% | 2.50 |
| {toothpaste} → {brush} | 25% | 62.5% | 2.50 |
| {milk, shampoo} → {cereals} | 25% | 100.0% | 2.50 |

## Interprétation

- **{mouthwash} → {toothpaste}**. Sur les 7 paniers avec du bain de bouche, 6 contiennent aussi du dentifrice (confiance de 85,7 %). Ces 6 paniers représentent 30 % de l’ensemble. Le dentifrice apparaît dans 40 % de tous les paniers, d’où un lift de 85,7 % ÷ 40 % ≈ 2,14.
- **{brush} → {toothpaste}**. Les 5 paniers avec une brosse contiennent tous du dentifrice (confiance de 100 %). Le support est de 25 % et le lift vaut 100 % ÷ 40 % = 2,50. Dans le sens inverse, le support reste de 25 % et la confiance descend à 62,5 %, car 5 des 8 paniers avec dentifrice contiennent une brosse.

Ces résultats décrivent un échantillon de 20 paniers.